# 問題
問題72で設計したモデルの重みベクトルを訓練セット上で学習せよ。ただし、学習中は単語埋め込み行列の値を固定せよ（単語埋め込み行列のファインチューニングは行わない）。また、学習時に損失値を表示するなど、学習の進捗状況をモニタリングできるようにせよ。

In [1]:
# 単語埋め込み語彙の作成
import numpy as np
from gensim.models import KeyedVectors

model = KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary=True)
vocab = list(model.key_to_index.keys())
d_emb = model.vector_size
V = len(vocab) + 1

# 埋め込み行列の初期化
E = np.zeros((V, d_emb), dtype=np.float32)

# インデックス対応表
word2id = {'<PAD>': 0}
id2word = {0: '<PAD>'}

# 行列にベクトルを格納
for i, word in enumerate(vocab, start=1):
    E[i] = model[word]
    word2id[word] = i
    id2word[i] = word

In [2]:
import torch

def sst_build_answer_list(path: str):
    """
    SST-2のTSVを読み込み、
      - 文章→単語分割
      - word2idでID列に変換（辞書にない語は除外）
      - パディングは行わず、可変長のTensorリストとして保持
    事前条件:
      - グローバルに `word2id` が存在
    返り値:
      - answer_list: List[Dict[str, Any]]
          text: 元文（str）
          label: torch.long テンソル（スカラ）
          input_ids: torch.long テンソル（長さ=文ごとに可変）
    """
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            row = raw.strip().split("\t")
            if len(row) < 2:
                continue
            label = row[1]
            if label not in ("0", "1"):
                continue
            text = row[0]
            tokens = text.strip().split()
            examples.append({"text": text, "tokens": tokens, "label": int(label)})

    # ID化（辞書外は除外）
    kept_examples = []
    for ex in examples:
        ids = [word2id[w] for w in ex["tokens"] if w in word2id]
        if ids:
            ex["input_ids"] = torch.tensor(ids, dtype=torch.long)
            kept_examples.append(ex)
        else:
            print(f'"{ex["text"]}" は辞書に該当単語がなくスキップしました')
            pass

    # answer_list生成
    answer_list = []
    for ex in kept_examples:
        answer_list.append({
            "text": ex["text"],
            "label": torch.tensor(ex["label"], dtype=torch.long),
            "input_ids": ex["input_ids"]
        })

    return answer_list

# --- pathの準備 ---
path_dev = "./SST-2/dev.tsv"
path_train = "./SST-2/train.tsv"
dev_71 = sst_build_answer_list(path_dev)
train_71 = sst_build_answer_list(path_train)

"oh-so-important " は辞書に該当単語がなくスキップしました
"beloved-major " は辞書に該当単語がなくスキップしました
"light-hearted " は辞書に該当単語がなくスキップしました
"time-consuming " は辞書に該当単語がなくスキップしました
"fresh-faced " は辞書に該当単語がなくスキップしました
"sleep-inducing " は辞書に該当単語がなくスキップしました
"self-absorbed " は辞書に該当単語がなくスキップしました
"good-natured " は辞書に該当単語がなくスキップしました
"queasy-stomached " は辞書に該当単語がなくスキップしました
"none-too-original " は辞書に該当単語がなくスキップしました
"well-intentioned " は辞書に該当単語がなくスキップしました
"big-hearted and " は辞書に該当単語がなくスキップしました
"well-meant " は辞書に該当単語がなくスキップしました
"kid-empowerment " は辞書に該当単語がなくスキップしました
"a rip-off " は辞書に該当単語がなくスキップしました
"therapy-dependent flakeball " は辞書に該当単語がなくスキップしました
"good-looking " は辞書に該当単語がなくスキップしました
"soon-to-be-forgettable " は辞書に該当単語がなくスキップしました
", cliche-ridden " は辞書に該当単語がなくスキップしました
"big-hearted " は辞書に該当単語がなくスキップしました
"'' has-been " は辞書に該当単語がなくスキップしました
"bad-movie " は辞書に該当単語がなくスキップしました
"linklater " は辞書に該当単語がなくスキップしました
"a re-hash " は辞書に該当単語がなくスキップしました
"well-put-together " は辞書に該当単語がなくスキップしました
"mind-numbing " は辞書に該当単語がなくスキップしました
"cheap-looking " は辞

In [3]:
import torch
from torch import nn
import numpy as np

def build_avg_features(examples, E, pad_id=0):
    # numpy配列ならtorch.Tensorに変換
    if isinstance(E, np.ndarray):
        E = torch.tensor(E, dtype=torch.float32)

    X, y = [], []
    for ex in examples:
        ids = ex["input_ids"]
        if pad_id is not None:
            ids = ids[ids != pad_id]
        if ids.numel() == 0:
            v = torch.zeros(E.size(1))
        else:
            v = E[ids].mean(dim=0)  # (len(ids), d) → (d,)
        X.append(v)
        y.append(ex["label"].float())

    return torch.stack(X, dim=0), torch.stack(y, dim=0)

X_train, y_train = build_avg_features(train_71, E)
X_dev, y_dev = build_avg_features(dev_71,   E)



In [5]:
import torch
from torch import nn

model = nn.Linear(X_train.size(1), 1)
crit  = nn.BCEWithLogitsLoss()
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ラベルをfloatに（BCEWithLogitsLoss用）
y_train_f = y_train.float()
y_dev_f   = y_dev.float()

epochs = 10
batch_size = 128

for epoch in range(1, epochs+1):
    model.train()
    # シャッフル
    idx = torch.randperm(X_train.size(0))
    X_train_shuf = X_train[idx]
    y_train_shuf = y_train_f[idx]

    running_loss = 0.0
    for i in range(0, X_train_shuf.size(0), batch_size):
        xb = X_train_shuf[i:i+batch_size]
        yb = y_train_shuf[i:i+batch_size]

        opt.zero_grad()
        logit = model(xb).squeeze(1)               # (B,)
        loss  = crit(logit, yb)
        loss.backward()
        opt.step()

        running_loss += loss.item() * xb.size(0)

    # ---- エポック終わりにログ ----
    train_loss = running_loss / X_train_shuf.size(0)

    model.eval()
    with torch.no_grad():
        logit_dev = model(X_dev).squeeze(1)
        prob_dev  = torch.sigmoid(logit_dev)
        pred_dev  = (prob_dev >= 0.5).long()
        acc_dev   = (pred_dev == y_dev.long()).float().mean().item()

    print(f"[Epoch {epoch:02d}] train loss: {train_loss:.4f}  |  dev acc: {acc_dev:.3f}")

# 学習後：重み・バイアス
w = model.weight.detach().squeeze(0)  # (d,)
b = model.bias.detach().item()

[Epoch 01] train loss: 0.5722  |  dev acc: 0.743
[Epoch 02] train loss: 0.4680  |  dev acc: 0.776
[Epoch 03] train loss: 0.4309  |  dev acc: 0.773
[Epoch 04] train loss: 0.4120  |  dev acc: 0.781
[Epoch 05] train loss: 0.4008  |  dev acc: 0.782
[Epoch 06] train loss: 0.3935  |  dev acc: 0.776
[Epoch 07] train loss: 0.3884  |  dev acc: 0.790
[Epoch 08] train loss: 0.3848  |  dev acc: 0.789
[Epoch 09] train loss: 0.3820  |  dev acc: 0.788
[Epoch 10] train loss: 0.3799  |  dev acc: 0.789


In [10]:
train_71

[{'text': 'hide new secretions from the parental units ',
  'label': tensor(0),
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags ',
  'label': tensor(0),
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature ',
  'label': tensor(1),
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,
           1964])},
 {'text': 'remains utterly satisfied to remain the same throughout ',
  'label': tensor(0),
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])},
 {'text': 'on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ',
  'label': tensor(0),
  'input_ids': tensor([    6,    12,  1445, 43789,    12, 10946,    76, 41349,    42])},
 {'text': "that 's far too tragic to merit such superficial treatment ",
  'label': tensor(0),


In [9]:
X_train

tensor([[ 0.1264,  0.0171, -0.0223,  ..., -0.0372,  0.0245,  0.0293],
        [ 0.1683, -0.0380,  0.1031,  ..., -0.1498, -0.0396,  0.0157],
        [ 0.0972,  0.0039,  0.0567,  ..., -0.0218,  0.0361, -0.0068],
        ...,
        [-0.0282,  0.0988,  0.0349,  ...,  0.0388,  0.0412, -0.0739],
        [ 0.1052,  0.1001, -0.0034,  ...,  0.0469,  0.1075,  0.0054],
        [ 0.0470,  0.0125,  0.0287,  ..., -0.1324,  0.0485, -0.0074]])